In [1]:
# pip install keras-tuner

In [18]:
import pandas as pd
import tensorflow
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv('huge_1M_titanic.csv')

In [3]:
data = data.sample(10000, random_state=42)

In [4]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
987231,988541,0,3,"Name988541, Mr. Surname988541",male,13.0,5,2,315084,44.567495,B57 B59 B63 B66,S
79954,81264,1,1,"Name81264, Miss. Surname81264",female,73.0,1,1,367655,133.025378,NaN,S
567130,568440,0,3,"Name568440, Mr. Surname568440",male,35.0,0,0,C.A. 5547,6.131359,NaN,S
500891,502201,1,1,"Name502201, Mrs. Surname502201",female,45.0,0,0,345764,141.897841,NaN,C
55399,56709,0,3,"Name56709, Mr. Surname56709",male,34.0,0,0,345774,19.546819,NaN,S


In [5]:
data = data.drop(columns = ['PassengerId','Name','Age','Ticket','Cabin'])

In [6]:
data['Embarked'] = data['Embarked'].replace({'S':'Southampton','C':'Chebourg','Q':'Queenstown'})

In [7]:
data.dropna(subset=['Embarked'],inplace = True)

In [8]:
data['Fare'] = data['Fare'].astype('int')

In [9]:
label = LabelEncoder()
onehot = OneHotEncoder(sparse_output = False)

In [10]:
data['Sex'] = label.fit_transform(data['Sex'])

In [11]:
Embarked = onehot.fit_transform(data[['Embarked']])
Embarked = pd.DataFrame(Embarked, columns = onehot.get_feature_names_out())
data = pd.concat([data.drop(columns = ['Embarked']),Embarked], axis=1)

In [12]:
scale = StandardScaler()

In [13]:
num_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']
data[num_cols] = scale.fit_transform(data[num_cols])

In [14]:
data = data.dropna()

In [15]:
X = data.drop(columns = ['Survived'])
y = data['Survived']

In [16]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 20, random_state = 42)

## Number of Optimal Hidden Layer

In [23]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape = (X_train.shape[1],)))

    for i in range(hp.Int('hidden', min_value=1, max_value=10, step=1)):
        model.add(Dense(88, activation = 'relu'))
    
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

    return model

In [25]:
tuner = kt.RandomSearch(build_model,objective = 'val_accuracy', max_trials = 5,directory = 'mydir')

In [26]:
tuner.search(X_train,y_train, validation_data = (X_test,y_test), epochs = 10)

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.6499999761581421

Best val_accuracy So Far: 0.800000011920929
Total elapsed time: 00h 00m 18s


In [27]:
tuner.get_best_hyperparameters()[0].values

{'hidden': 3}

In [29]:
model = tuner.get_best_models(num_models=1)[0]

c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [31]:
model.fit(X_train,y_train, validation_data=(X_test,y_test), epochs = 100, batch_size = 32)

Epoch 1/100


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9286 - loss: 0.1822 - val_accuracy: 0.6500 - val_loss: 2.0583
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9286 - loss: 0.1787 - val_accuracy: 0.6500 - val_loss: 2.0706
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9286 - loss: 0.1765 - val_accuracy: 0.6500 - val_loss: 2.1165
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9286 - loss: 0.1765 - val_accuracy: 0.6500 - val_loss: 2.1571
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9286 - loss: 0.1754 - val_accuracy: 0.6500 - val_loss: 2.1432
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9286 - loss: 0.1743 - val_accuracy: 0.6500 - val_loss: 2.1219
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9286 - loss: 0.1764 - val_accuracy: 0.6500 - val_loss: 2.1288
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.9286 - loss: 0.1804 - val_accuracy: 0.6500 - val_loss: 2.2046
Epoc